# Deploy MCP Servers on OpenShift

This notebook deploys 5 MCP servers as shared services on OpenShift, accessible by all team members via Routes.

**Servers to deploy:**
1. Context7 — Library documentation (remote, no deployment needed)
2. Sequential Thinking — Structured problem solving
3. GitHub — Repository operations
4. gh-grep — GitHub code search
5. Chrome DevTools — Browser automation

## 1. Verify Cluster Access

In [ ]:
%%bash
echo "Cluster: $(oc whoami --show-server)"
echo "User: $(oc whoami)"
echo ""
echo "Apps domain (for Route URLs):"
oc get ingresses.config cluster -o jsonpath='{.spec.domain}'
echo ""

## 2. Create Namespace and Secrets

All MCP servers deploy into the `mcp-servers` namespace.

In [ ]:
%%bash
# Create namespace
oc apply -f manifests/00-namespace-secret.yaml

echo ""
echo "⚠️  IMPORTANT: Update the secret with your actual tokens:"
echo ""
echo "  oc set data secret/mcp-api-keys -n mcp-servers \\"
echo "    --from-literal=GITHUB_TOKEN=ghp_your-actual-token"

In [ ]:
import os
from dotenv import load_dotenv
import subprocess

load_dotenv("../.env")

github_token = os.getenv("GITHUB_TOKEN")

if github_token and github_token != "ghp_your-github-token-here":
    subprocess.run([
        "oc", "create", "secret", "generic", "mcp-api-keys",
        f"--from-literal=GITHUB_TOKEN={github_token}",
        "-n", "mcp-servers",
        "--dry-run=client", "-o", "yaml"
    ], capture_output=False)
    # Pipe to oc apply if you want auto-update:
    # | oc apply -f -
    print("✅ Tokens loaded from .env")
else:
    print("⚠️  Set GITHUB_TOKEN in .env, then re-run or use oc command above.")

## 3. Server 1 — Context7 (Remote, No Deploy)

Context7 is an externally hosted MCP server at `https://mcp.context7.com/mcp`.

No deployment needed — IDEs connect directly.

**Tools:** `resolve-library-id`, `get-library-docs`

In [ ]:
%%bash
# Verify Context7 is reachable from the cluster
echo "Testing Context7 connectivity..."
HTTP_CODE=$(curl -s -o /dev/null -w "%{http_code}" https://mcp.context7.com/mcp)

if [ "$HTTP_CODE" = "200" ] || [ "$HTTP_CODE" = "405" ]; then
    echo "✅ Context7 reachable (HTTP $HTTP_CODE)"
    echo "   Endpoint: https://mcp.context7.com/mcp"
    echo "   No deployment needed."
else
    echo "❌ Context7 unreachable (HTTP $HTTP_CODE)"
fi

## 4. Server 2 — Sequential Thinking

Wraps `@modelcontextprotocol/server-sequential-thinking` (stdio) with `supergateway` to expose as HTTP SSE.

In [ ]:
%%bash
echo "Deploying Sequential Thinking MCP server..."
oc apply -f manifests/01-sequential-thinking.yaml

echo ""
echo "Waiting for pod to be ready..."
oc wait --for=condition=available deployment/mcp-sequential-thinking -n mcp-servers --timeout=120s

echo ""
echo "Route URL:"
oc get route mcp-sequential-thinking -n mcp-servers -o jsonpath='https://{.spec.host}/sse'
echo ""

## 5. Server 3 — GitHub MCP

Full GitHub API access. Requires `GITHUB_TOKEN` in the Secret.

In [ ]:
%%bash
echo "Deploying GitHub MCP server..."
oc apply -f manifests/02-github.yaml

echo ""
echo "Waiting for pod to be ready..."
oc wait --for=condition=available deployment/mcp-github -n mcp-servers --timeout=120s

echo ""
echo "Route URL:"
oc get route mcp-github -n mcp-servers -o jsonpath='https://{.spec.host}/sse'
echo ""

## 6. Server 4 — gh-grep (Custom Server)

Custom MCP server for GitHub code search. Server code is stored in a ConfigMap.

In [ ]:
%%bash
echo "Deploying gh-grep MCP server..."
oc apply -f manifests/03-gh-grep.yaml

echo ""
echo "Waiting for pod to be ready..."
oc wait --for=condition=available deployment/mcp-gh-grep -n mcp-servers --timeout=120s

echo ""
echo "Route URL:"
oc get route mcp-gh-grep -n mcp-servers -o jsonpath='https://{.spec.host}/sse'
echo ""

## 7. Server 5 — Chrome DevTools

Browser automation with bundled Chromium. Uses more resources due to headless Chrome.

In [ ]:
%%bash
echo "Deploying Chrome DevTools MCP server..."
oc apply -f manifests/04-chrome-devtools.yaml

echo ""
echo "Waiting for pod to be ready (may take longer due to Chromium)..."
oc wait --for=condition=available deployment/mcp-chrome-devtools -n mcp-servers --timeout=180s

echo ""
echo "Route URL:"
oc get route mcp-chrome-devtools -n mcp-servers -o jsonpath='https://{.spec.host}/sse'
echo ""

## 8. Verify All Servers

In [ ]:
%%bash
echo "MCP Server Deployment Status"
echo "============================================================"
echo ""
echo "=== Pods ==="
oc get pods -n mcp-servers -o wide

echo ""
echo "=== Routes (IDE Endpoints) ==="
echo ""
printf "%-25s %s\n" "SERVER" "ENDPOINT"
printf "%-25s %s\n" "-------" "--------"
printf "%-25s %s\n" "Context7 (remote)" "https://mcp.context7.com/mcp"

for route in $(oc get routes -n mcp-servers -o jsonpath='{.items[*].metadata.name}'); do
    host=$(oc get route $route -n mcp-servers -o jsonpath='{.spec.host}')
    printf "%-25s %s\n" "$route" "https://${host}/sse"
done

In [ ]:
import subprocess

# Health check all routes
result = subprocess.run(
    ["oc", "get", "routes", "-n", "mcp-servers",
     "-o", "jsonpath={range .items[*]}{.metadata.name}={.spec.host}\n{end}"],
    capture_output=True, text=True
)

print("Health Check:")
print("=" * 60)

# Check Context7
r = subprocess.run(["curl", "-sk", "-o", "/dev/null", "-w", "%{http_code}", "-m", "5",
                    "https://mcp.context7.com/mcp"], capture_output=True, text=True)
status = "✅" if r.stdout.strip() in ["200", "405"] else "❌"
print(f"{status} Context7 (remote): https://mcp.context7.com/mcp")

# Check deployed servers
for line in result.stdout.strip().split("\n"):
    if "=" in line:
        name, host = line.split("=", 1)
        url = f"https://{host}/sse"
        r = subprocess.run(["curl", "-sk", "-o", "/dev/null", "-w", "%{http_code}", "-m", "5", url],
                          capture_output=True, text=True)
        status = "✅" if r.stdout.strip() in ["200", "405"] else "❌"
        print(f"{status} {name}: {url}")

## Summary

| Server | Deployment | Route |
|--------|-----------|-------|
| Context7 | Remote (no deploy) | `https://mcp.context7.com/mcp` |
| Sequential Thinking | OpenShift Pod | `https://mcp-sequential-thinking-mcp-servers.apps.CLUSTER/sse` |
| GitHub | OpenShift Pod | `https://mcp-github-mcp-servers.apps.CLUSTER/sse` |
| gh-grep | OpenShift Pod | `https://mcp-gh-grep-mcp-servers.apps.CLUSTER/sse` |
| Chrome DevTools | OpenShift Pod | `https://mcp-chrome-devtools-mcp-servers.apps.CLUSTER/sse` |

## Next Steps

→ `3_connect_ide_clients.ipynb` — Configure your IDE to use these MCP server Routes (direct access)
→ `../2_ai_gateway/2_enable_maas.ipynb` — Register MCP servers with MaaS gateway for unified access with auth